# 🏠 California Housing Price Prediction using ANN

This notebook builds an **Artificial Neural Network (ANN)** regression model to predict median house values in California using the [California Housing Dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html).

## Pipeline Overview
1. Load & explore the dataset
2. Preprocess features (train/test split + standardization)
3. Build & compile the ANN model
4. Train with validation monitoring
5. Evaluate with MAE and R² score
6. Visualize training curves and predictions

---

## 1. Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
import warnings

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')

## 2. Load & Explore Dataset

The **California Housing Dataset** contains 20,640 samples with 8 features derived from the 1990 US Census.

| Feature | Description |
|---|---|
| `MedInc` | Median income in block group |
| `HouseAge` | Median house age in block group |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block group population |
| `AveOccup` | Average household size |
| `Latitude` | Block group latitude |
| `Longitude` | Block group longitude |
| `MedHouseVal` | **Target** — Median house value (in $100,000s) |

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
df.describe().round(3)

## 3. Preprocessing

- **Features (X):** All columns except the target
- **Target (y):** `MedHouseVal` (median house value in $100,000s)
- **Split:** 80% train / 20% test
- **Scaling:** `StandardScaler` fitted on training data only (prevents data leakage)

In [ ]:
X = df.drop(columns=['MedHouseVal']).values
y = df['MedHouseVal'].values

print(f'Features shape : {X.shape}')
print(f'Target shape   : {y.shape}')
print(f'Target range   : [{y.min():.2f}, {y.max():.2f}] ($100k)')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)  # fit + transform on train
X_test  = scaler.transform(X_test)       # transform only on test

print(f'Train samples : {X_train.shape[0]}')
print(f'Test samples  : {X_test.shape[0]}')

## 4. Build the ANN Model

A 3-layer fully connected network:

```
Input(8) → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Linear)
```

- **Hidden layers:** ReLU activation for non-linearity
- **Output layer:** Linear activation (unbounded regression output)
- **Loss:** MSE (Mean Squared Error)
- **Optimizer:** Adam

In [ ]:
model = Sequential([
    Input(shape=(8,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1,  activation='linear')
], name='ANN_regression')

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

## 5. Train the Model

Training for **50 epochs** with:
- Batch size of 32
- 10% of training data held out for validation

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

## 6. Evaluation

Measuring performance on the held-out test set using:
- **MAE** — Mean Absolute Error (in $100,000s)
- **R²** — Coefficient of Determination (1.0 = perfect fit)

In [ ]:
y_pred = model.predict(X_test).flatten()

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print('=' * 35)
print('       TEST SET RESULTS')
print('=' * 35)
print(f'  MAE : {mae:.4f}  (~${mae*100_000:,.0f})')
print(f'  R²  : {r2:.4f}')
print('=' * 35)

## 7. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ANN Regression — California Housing', fontsize=14, fontweight='bold')

# --- Plot 1: Training & Validation Loss (MSE) ---
ax = axes[0]
ax.plot(history.history['loss'],     label='Train Loss',      color='steelblue')
ax.plot(history.history['val_loss'], label='Val Loss',        color='tomato', linestyle='--')
ax.set_title('Loss (MSE) over Epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.legend()
ax.grid(alpha=0.3)

# --- Plot 2: Training & Validation MAE ---
ax = axes[1]
ax.plot(history.history['mae'],     label='Train MAE', color='steelblue')
ax.plot(history.history['val_mae'], label='Val MAE',   color='tomato', linestyle='--')
ax.set_title('MAE over Epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('MAE')
ax.legend()
ax.grid(alpha=0.3)

# --- Plot 3: Actual vs Predicted ---
ax = axes[2]
ax.scatter(y_test, y_pred, alpha=0.3, s=10, color='steelblue', label='Predictions')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Fit')
ax.set_title(f'Actual vs Predicted  (R² = {r2:.3f})')
ax.set_xlabel('Actual Value ($100k)')
ax.set_ylabel('Predicted Value ($100k)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as results.png')